# babybath.babybath_dict — EDA for Embedding Feature Importance

Goal: profile every column, identify which carry signal for embeddings, and rank them by likely importance. Not a model — just the numbers that tell you where to point the embedding.

## 1. Setup

In [1]:
!pip install -q google-cloud-bigquery db-dtypes scikit-learn

In [2]:
import os
from google.colab import auth, files

# uploaded = files.upload()
# if uploaded:
#     creds_path = list(uploaded.keys())[0]
#     os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = creds_path
#     print(f'Using service account: {creds_path}')
# else:
#     pass
auth.authenticate_user()
print('Using Colab OAuth')
print('ok')

Using Colab OAuth
ok


In [10]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = "sincere-hearth-273704"  # adjust if different
DATASET = "babybath"
TABLE = "babybath_dict"

client = bigquery.Client(project=PROJECT)
print(f"-> {PROJECT}.{DATASET}.{TABLE}")

-> sincere-hearth-273704.babybath.babybath_dict


In [ ]:
# Load the full table into a DataFrame ONCE — all subsequent analysis uses pandas, not BQ
df = client.query(f"SELECT * FROM `{PROJECT}.{DATASET}.{TABLE}`").to_dataframe()
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head(3)

## 2. Schema & row count

In [18]:
# Infer schema from the DataFrame
schema = pd.DataFrame({
    'column_name': df.columns,
    'data_type': ['FLOAT64' if str(d) == 'float64' else 'STRING' for d in df.dtypes]
})
n_rows = len(df)
print(f"{len(schema)} columns, {n_rows:,} rows")
schema

## 3. Sample rows

In [14]:
df.head(10)

## 4. NULL profile

In [24]:
# Compute NULL percentages from the in-memory DataFrame
# Treat empty strings as NULL for STRING columns
null_pct = pd.Series(dtype=float)
for col in df.columns:
    if col == 'bundle_quantity':
        null_pct[f'{col}_null'] = df[col].isna().sum() / len(df) * 100
    else:
        null_pct[f'{col}_null'] = ((df[col].isna()) | (df[col] == '')).sum() / len(df) * 100
null_pct = null_pct.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(schema) * 0.3)))
null_pct.plot(kind='barh', ax=ax, color='#C44E52')
ax.set_xlabel('% NULL')
ax.set_title(f'NULL % per column (n={n_rows:,})')
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
null_pct

## 5. Cardinality

In [20]:
# Compute cardinality from the in-memory DataFrame
card_series = df.nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(schema) * 0.3)))
card_series.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_xlabel('Distinct values')
ax.set_title('Cardinality per column')
plt.tight_layout()
plt.show()

# Build card_df for downstream use (columns: column, distinct, pct_unique)
card_df = card_series.reset_index()
card_df.columns = ['column', 'distinct']
card_df['pct_unique'] = (card_df['distinct'] / n_rows * 100).round(2)
card_df

## 5.5. Cardinality vs NULL scatter — the one chart that matters

Top-right = high signal, low noise. Bottom-left = skip.

In [28]:
scatter_df = card_df.copy()
# Map null_pct by matching column names: card_df has 'column' = e.g. 'sku_type', null_pct has 'sku_type_null'
scatter_df['null_pct'] = scatter_df['column'].apply(
    lambda c: null_pct.get(f'{c}_null', null_pct.get(f'{c}_distinct_null', 100))
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scatter_df['pct_unique'], 100 - scatter_df['null_pct'],
           s=100, c='#4C72B0', alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('Unique value %  (-> more signal)')
ax.set_ylabel('Completeness %  (-> less NULL)')
ax.set_title('Cardinality vs completeness — each dot = one column')
ax.set_xlim(-2, 105)
ax.set_ylim(-2, 105)

top = scatter_df.nlargest(8, 'pct_unique')
for _, r in top.iterrows():
    ax.annotate(r['column'], (r['pct_unique'], 100 - r['null_pct']),
                fontsize=8, xytext=(5, 3), textcoords='offset points')

# quadrant shading
ax.axhline(70, color='gray', linestyle=':', alpha=0.3)
ax.axvline(10, color='gray', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

scatter_df['signal_tier'] = 'low'
scatter_df.loc[(scatter_df['pct_unique'] > 10) & ((100 - scatter_df['null_pct']) > 70), 'signal_tier'] = 'medium'
scatter_df.loc[(scatter_df['pct_unique'] > 30) & ((100 - scatter_df['null_pct']) > 90), 'signal_tier'] = 'high'
scatter_df[['column', 'pct_unique', 'null_pct', 'signal_tier']].sort_values('pct_unique', ascending=False)

## 6. String columns — length distribution

In [30]:
str_cols = schema[schema['data_type'].str.upper().str.contains('STRING')]['column_name'].tolist()
print(f"String columns: {str_cols}")

if str_cols:
    # Compute p50 and p95 string lengths from the DataFrame
    rows = []
    for col in str_cols:
        lens = df[col].dropna().str.len()
        if len(lens) > 0:
            rows.append({'column': col, 'p50_len': lens.quantile(0.50), 'p95_len': lens.quantile(0.95)})
        else:
            rows.append({'column': col, 'p50_len': 0, 'p95_len': 0})
    len_df = pd.DataFrame(rows)

    # Build a lookup dict for the scoring cell (keys are column names)
    lengths = pd.DataFrame({
        f'{r["column"]}_p50': [r['p50_len']] for r in rows
    } | {
        f'{r["column"]}_p95': [r['p95_len']] for r in rows
    })

    fig, ax = plt.subplots(figsize=(8, max(3, len(str_cols) * 0.4)))
    x = range(len(str_cols))
    ax.barh(x, len_df['p50_len'], color='#55A868', label='p50')
    ax.barh(x, len_df['p95_len'] - len_df['p50_len'],
            left=len_df['p50_len'], color='#DD8452', alpha=0.6, label='p95-p50')
    ax.set_yticks(x)
    ax.set_yticklabels(str_cols)
    ax.set_xlabel('String length (chars)')
    ax.legend()
    ax.set_title('String length distribution (p50, p95)')
    plt.tight_layout()
    plt.show()
    len_df

## 7. Numeric columns — distribution shape

In [31]:
num_cols = schema[schema['data_type'].str.upper().str.contains('INT|FLOAT|NUMERIC')]['column_name'].tolist()
print(f"Numeric columns: {num_cols}")

if num_cols:
    rows = []
    for col in num_cols:
        s = df[col].dropna()
        rows.append({
            'column': col,
            'min': s.min(),
            'p50': s.quantile(0.50),
            'avg': s.mean(),
            'p95': s.quantile(0.95),
            'max': s.max(),
        })
    num_stats = pd.DataFrame(rows)
    num_stats

## 7.5. Numeric histograms (20 bins)

Skewed = poor embedding input. Flat-ish = more signal per dimension.

In [38]:
if num_cols:
    n_num = len(num_cols)
    cols_per_row = min(n_num, 4)
    n_rows_plot = (n_num + cols_per_row - 1) // cols_per_row

    fig, axes = plt.subplots(n_rows_plot, cols_per_row,
                             figsize=(4 * cols_per_row, 3.5 * n_rows_plot))
    if n_num == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, col in enumerate(num_cols):
        ax = axes[idx]
        series = df[col].dropna()
        if len(series) < 2:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(col)
            continue
        ax.hist(series, bins=20, color='#55A868', alpha=0.85, edgecolor='white', linewidth=0.3)
        ax.set_title(col, fontsize=10)

    for idx in range(n_num, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle('Numeric column distributions (20 equal-width bins)', y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()

## 8. String value overlap — pairwise Jaccard

In [34]:
from sklearn.feature_extraction.text import CountVectorizer

if len(str_cols) >= 2:
    # Sample up to 5000 rows from the DataFrame
    sample = df[[c for c in str_cols]].dropna(how='all').head(5000)

    n = len(str_cols)
    jaccard = np.zeros((n, n))
    for i, a in enumerate(str_cols):
        for j, b in enumerate(str_cols):
            if j <= i:
                continue
            set_a = set(sample[a].dropna().astype(str).str.lower())
            set_b = set(sample[b].dropna().astype(str).str.lower())
            jac = len(set_a & set_b) / max(len(set_a | set_b), 1)
            jaccard[i, j] = jaccard[j, i] = jac

    jac_df = pd.DataFrame(jaccard, index=str_cols, columns=str_cols)

    fig, ax = plt.subplots(figsize=(max(5, n * 0.9), max(4, n * 0.8)))
    im = ax.imshow(jaccard, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(str_cols, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(str_cols, fontsize=8)
    for i in range(n):
        for j in range(n):
            if j > i:
                ax.text(j, i, f'{jaccard[i, j]:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if jaccard[i, j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, label='Jaccard', shrink=0.8)
    ax.set_title('String column value overlap (Jaccard, n=5000)')
    plt.tight_layout()
    plt.show()
    jac_df.round(3)

## 8.5. Value concentration — how evenly are values distributed?

A column where the top-5 values cover 90% of rows is a label, not an embedding target. A column with a long tail (top-50 < 20%) is embedding-rich.

In [48]:
# pick top-6 scoring string columns for concentration plot
top_str = [c for c in card_df.sort_values('pct_unique', ascending=False)['column']
           if c in str_cols][:6]

if top_str:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()

    for ax, col in zip(axes, top_str):
        freq = df[col].dropna().value_counts().reset_index()
        freq.columns = ['val', 'n']

        if freq.empty:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(col)
            continue

        freq['cum_pct'] = freq['n'].cumsum() / freq['n'].sum() * 100
        freq['rank'] = range(1, len(freq) + 1)
        plot_n = min(50, len(freq))
        ax.plot(freq['rank'][:plot_n], freq['cum_pct'][:plot_n], color='#4C72B0', linewidth=1.5)
        ax.fill_between(freq['rank'][:plot_n], freq['cum_pct'][:plot_n], alpha=0.15, color='#4C72B0')
        ax.axhline(80, color='#C44E52', linestyle='--', alpha=0.5, linewidth=0.8)
        r80 = freq[freq['cum_pct'] >= 80]['rank'].min()
        ax.set_title(f"{col}\n80% covered by top-{int(r80)} values", fontsize=9)
        ax.set_xlabel('Value rank')
        ax.set_ylabel('Cumulative % of rows')

    for idx in range(len(top_str), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle('Value concentration curves — steep = few values dominate, shallow = diverse', y=1.01)
    plt.tight_layout()
    plt.show()

## 9. Embedding signal score — compound heuristic

diversity * completeness * length_fitness. High cardinality + mid-length strings + low NULL = high score.

In [49]:
score_rows = []
for _, row in schema.iterrows():
    col = row['column_name']
    dtype = row['data_type'].upper()
    card = card_series.get(col, 0)
    null_p = null_pct.get(f'{col}_null', 100)
    is_str = 'STRING' in dtype

    diversity = min(card / max(n_rows, 1), 1.0)
    completeness = 1 - (null_p / 100)

    if is_str and col in str_cols:
        p50_col = f'{col}_p50'
        p95_col = f'{col}_p95'
        if p50_col in lengths.columns:
            p50 = lengths[p50_col].iloc[0]
            p95 = lengths[p95_col].iloc[0]
        else:
            p50 = p95 = 0
        if p50 < 3:
            len_score = 0.2
        elif p50 > 500:
            len_score = 0.3
        else:
            len_score = 1.0
    else:
        p50 = p95 = len_score = None

    score = diversity * completeness * (len_score if is_str else 0.1)
    score_rows.append({
        'column': col,
        'dtype': dtype,
        'distinct': card,
        'null_pct': round(null_p, 1),
        'p50_len': p50,
        'p95_len': p95,
        'embedding_score': round(score, 4),
    })

ranking = pd.DataFrame(score_rows).sort_values('embedding_score', ascending=False)

fig, ax = plt.subplots(figsize=(8, max(3, len(ranking) * 0.35)))
colors = ['#55A868' if s > 0.3 else '#DD8452' if s > 0.1 else '#C44E52'
          for s in ranking['embedding_score']]
ax.barh(ranking['column'], ranking['embedding_score'], color=colors)
ax.set_xlabel('Embedding signal score')
ax.set_title('Which columns to embed (heuristic)')
plt.tight_layout()
plt.show()
ranking

## 10. Top-N frequent values — content check

In [51]:
top_cols = ranking[ranking['dtype'].str.contains('STRING')].head(3)['column'].tolist()

for col in top_cols:
    freq = df[col].dropna().value_counts().head(15).reset_index()
    freq.columns = ['val', 'n']
    print(f"\n--- {col} (top 15) ---")
    display(freq)

## Interpretation guide

- **Cardinality-vs-NULL scatter**: top-right quadrant = embed candidates. High unique% + high completeness.
- **Value concentration curves**: steep (top-5 covers >80%) = label/category, not embedding material. Shallow = good.
- **Jaccard heatmap**: pairs >0.7 = redundant. Keep the one with better cardinality/length.
- **Numeric histograms**: heavily skewed = bucket it or skip. Bimodal/flat = may carry signal.
- **embedding_score > 0.3**: embed these. 0.1-0.3: concatenate as context. < 0.1: skip.
- **> 50% NULL**: exclude from embeddings.